# EF1 Agent for the 7-4-1 Bargaining Game

This notebook walks through the design and behavior of a rule-based negotiation agent that uses **Envy-Free up to 1 item (EF1)** fairness as its core strategy.

## Game Setup

Two players divide items with quantities **[7, 4, 1]** (12 total items across 3 types). Each player has:
- **Private values** (1-100) for each item type
- A **private outside option** (walk-away value)

The game lasts up to 3 rounds. On each turn a player can **counteroffer**, **accept**, or **walk**.

## Agent Strategy

1. Find all allocations where the agent satisfies **EF1**
2. Among those, find the **Pareto frontier** of offers giving the opponent the most items
3. **Open** with the most generous frontier offer (max total items to opponent)
4. **Accept** any counteroffer that is EF1 and beats the outside option
5. **Counter** with a different frontier offer if the opponent's offer is not EF1
6. **Walk** if no EF1 deal beats the outside option

In [37]:
! pip install torch

In [38]:
from ef1_agent import EF1Agent, QUANTITIES, NUM_ITEM_TYPES


## Step 1: What is EF1?

An allocation is **Envy-Free up to 1 item (EF1)** for a player if they don't envy the opponent's bundle after mentally removing one item from it.

Formally, given the agent's values $v$, the agent's bundle $A$, and the opponent's bundle $B$:

$$v(A) \geq v(B) - \max_{j : B_j > 0} v_j$$

where $v_j$ is the per-unit value of item type $j$. The agent removes the single most valuable unit (from their own perspective) in the opponent's bundle.

In [39]:
agent = EF1Agent()

# Example: values [80, 20, 50] per unit
values = [80.0, 20.0, 50.0]

# Agent keeps (4, 2, 1), opponent gets (3, 2, 0)
my = (4, 2, 1)
opp = (3, 2, 0)
my_val = EF1Agent._value_of(values, my)   # 4*80 + 2*20 + 1*50 = 410
opp_val = EF1Agent._value_of(values, opp)  # 3*80 + 2*20 + 0*50 = 280

print(f"My bundle: {my} -> value = {my_val}")
print(f"Opp bundle: {opp} -> value (from my perspective) = {opp_val}")
print(f"Envy-free? {my_val >= opp_val} (no envy at all)")
print(f"EF1? {EF1Agent._is_ef1(values, my, opp)}")
print()

My bundle: (4, 2, 1) -> value = 410.0
Opp bundle: (3, 2, 0) -> value (from my perspective) = 280.0
Envy-free? True (no envy at all)
EF1? True



In [40]:
# A case where EF1 matters: agent has fewer valuable items
my2 = (3, 4, 1)
opp2 = (4, 0, 0)
my_val2 = EF1Agent._value_of(values, my2)   # 3*80 + 4*20 + 1*50 = 370
opp_val2 = EF1Agent._value_of(values, opp2)  # 4*80 = 320

print(f"My bundle: {my2} -> value = {my_val2}")
print(f"Opp bundle: {opp2} -> value (from my perspective) = {opp_val2}")
print(f"Envy-free? {my_val2 >= opp_val2}")
print(f"EF1? {EF1Agent._is_ef1(values, my2, opp2)}")
print()

# A case that fails EF1
my3 = (2, 4, 1)
opp3 = (5, 0, 0)
my_val3 = EF1Agent._value_of(values, my3)   # 2*80 + 4*20 + 1*50 = 290
opp_val3 = EF1Agent._value_of(values, opp3)  # 5*80 = 400
max_removal = max(values[j] for j in range(3) if opp3[j] > 0)  # 80 (item 0)

print(f"My bundle: {my3} -> value = {my_val3}")
print(f"Opp bundle: {opp3} -> value (from my perspective) = {opp_val3}")
print(f"Envy = {opp_val3} - {my_val3} = {opp_val3 - my_val3}")
print(f"Max removable (1 unit of most valuable type opp holds) = {max_removal}")
print(f"Envy after removal = {opp_val3 - my_val3 - max_removal}")
print(f"EF1? {EF1Agent._is_ef1(values, my3, opp3)} (still envious after removing 1 item)")

My bundle: (3, 4, 1) -> value = 370.0
Opp bundle: (4, 0, 0) -> value (from my perspective) = 320.0
Envy-free? True
EF1? True

My bundle: (2, 4, 1) -> value = 290.0
Opp bundle: (5, 0, 0) -> value (from my perspective) = 400.0
Envy = 400.0 - 290.0 = 110.0
Max removable (1 unit of most valuable type opp holds) = 80.0
Envy after removal = 30.0
EF1? False (still envious after removing 1 item)


## Step 2: Finding All EF1 Allocations

With item quantities [7, 4, 1], there are only **80 possible allocations** (8 x 5 x 2). We enumerate all of them and filter to those satisfying EF1.

In [41]:
values = [80.0, 20.0, 50.0]
ef1_allocs = agent._find_ef1_allocations(values)

print(f"Total possible allocations: {len(agent._all_allocations)}")
print(f"EF1 allocations: {len(ef1_allocs)}")
print(f"Non-EF1 allocations: {len(agent._all_allocations) - len(ef1_allocs)}")
print()
print("Sample EF1 allocations (agent keeps, opponent gets, agent value, items to opp):")
for my, opp in sorted(ef1_allocs, key=lambda x: sum(x[1]), reverse=True)[:10]:
    print(f"  keep={my}  give={opp}  my_val={EF1Agent._value_of(values, my):>5.0f}  total_to_opp={sum(opp)}")

Total possible allocations: 80
EF1 allocations: 45
Non-EF1 allocations: 35

Sample EF1 allocations (agent keeps, opponent gets, agent value, items to opp):
  keep=(4, 0, 0)  give=(3, 4, 1)  my_val=  320  total_to_opp=8
  keep=(3, 1, 1)  give=(4, 3, 0)  my_val=  310  total_to_opp=7
  keep=(4, 0, 1)  give=(3, 4, 0)  my_val=  370  total_to_opp=7
  keep=(4, 1, 0)  give=(3, 3, 1)  my_val=  340  total_to_opp=7
  keep=(5, 0, 0)  give=(2, 4, 1)  my_val=  400  total_to_opp=7
  keep=(3, 2, 1)  give=(4, 2, 0)  my_val=  330  total_to_opp=6
  keep=(4, 1, 1)  give=(3, 3, 0)  my_val=  390  total_to_opp=6
  keep=(4, 2, 0)  give=(3, 2, 1)  my_val=  360  total_to_opp=6
  keep=(5, 0, 1)  give=(2, 4, 0)  my_val=  450  total_to_opp=6
  keep=(5, 1, 0)  give=(2, 3, 1)  my_val=  420  total_to_opp=6


## Step 3: The Pareto Frontier

Among EF1 allocations, we find the **Pareto frontier of generosity**: offers where we can't give the opponent more of *any* item type without either giving less of another type or violating EF1.

An offer $A$ **dominates** offer $B$ if $A_{opp}[i] \geq B_{opp}[i]$ for all item types $i$ and strictly greater for at least one.

After computing the frontier, we **filter out any allocation where the agent's share doesn't strictly beat its outside option**. This prevents the agent from proposing deals that leave it worse off than walking.

In [42]:
frontier = EF1Agent._pareto_frontier(ef1_allocs)
sorted_frontier = EF1Agent._sort_frontier(frontier)

print(f"Pareto frontier size (before BATNA filter): {len(sorted_frontier)}")
print()
print("Full frontier offers:")
print(f"{'Idx':>3}  {'Keep':>12}  {'Give':>12}  {'Total':>5}  {'Frac given':>20}  {'My Value':>8}")
print("-" * 70)
for i, (my, opp) in enumerate(sorted_frontier):
    fracs = tuple(round(opp[j] / QUANTITIES[j], 2) for j in range(3))
    print(f"  {i}  {str(my):>12}  {str(opp):>12}  {sum(opp):>5}  {str(fracs):>20}  {EF1Agent._value_of(values, my):>8.0f}")

# Filter to only allocations where agent's share strictly beats outside option
outside_option = 300.0
sorted_frontier_filtered = [
    (my, opp) for my, opp in sorted_frontier
    if EF1Agent._value_of(values, my) > outside_option + 1e-6
]

print(f"\nAfter BATNA filter (outside option = {outside_option}):")
print(f"Frontier size: {len(sorted_frontier)} -> {len(sorted_frontier_filtered)}")
print()
for i, (my, opp) in enumerate(sorted_frontier_filtered):
    fracs = tuple(round(opp[j] / QUANTITIES[j], 2) for j in range(3))
    print(f"  {i}  {str(my):>12}  {str(opp):>12}  {sum(opp):>5}  {str(fracs):>20}  {EF1Agent._value_of(values, my):>8.0f}")

Pareto frontier size (before BATNA filter): 3

Full frontier offers:
Idx          Keep          Give  Total            Frac given  My Value
----------------------------------------------------------------------
  0     (4, 0, 0)     (3, 4, 1)      8      (0.43, 1.0, 1.0)       320
  1     (3, 1, 1)     (4, 3, 0)      7     (0.57, 0.75, 0.0)       310
  2     (3, 4, 0)     (4, 0, 1)      5      (0.57, 0.0, 1.0)       320

After BATNA filter (outside option = 300.0):
Frontier size: 3 -> 3

  0     (4, 0, 0)     (3, 4, 1)      8      (0.43, 1.0, 1.0)       320
  1     (3, 1, 1)     (4, 3, 0)      7     (0.57, 0.75, 0.0)       310
  2     (3, 4, 0)     (4, 0, 1)      5      (0.57, 0.0, 1.0)       320


## Step 4: Opening Offer

The agent opens with the filtered frontier offer that gives the opponent the **most total items**. This is the most generous EF1-fair offer that still beats the agent's outside option.

In [43]:
opening_my, opening_opp = sorted_frontier[0]
print(f"Values: {values}")
print(f"Opening offer: give {opening_opp} to opponent (keep {opening_my})")
print(f"Total items to opponent: {sum(opening_opp)} out of {sum(QUANTITIES)}")
print(f"Agent's value: {EF1Agent._value_of(values, opening_my):.0f}")
print(f"Opponent's value (from agent's view): {EF1Agent._value_of(values, opening_opp):.0f}")
print()

# Verify it's EF1
my_v = EF1Agent._value_of(values, opening_my)
opp_v = EF1Agent._value_of(values, opening_opp)
max_rem = max(values[j] for j in range(3) if opening_opp[j] > 0)
print(f"EF1 check: {my_v} >= {opp_v} - {max_rem} = {opp_v - max_rem}? {my_v >= opp_v - max_rem}")

Values: [80.0, 20.0, 50.0]
Opening offer: give (3, 4, 1) to opponent (keep (4, 0, 0))
Total items to opponent: 8 out of 12
Agent's value: 320
Opponent's value (from agent's view): 370

EF1 check: 320.0 >= 370.0 - 80.0 = 290.0? True


## Step 5: Cycling Through Offers

If the opponent rejects with a non-EF1 counteroffer, the agent picks a **different** frontier offer that emphasizes different item types. The frontier is sorted so consecutive offers shift which item type is given most.

The agent indexes into the filtered frontier by round number, so each round offers a different composition. All offers in the cycle are guaranteed to be EF1-fair and above the agent's outside option.

In [44]:
print("Cycling through rounds (using BATNA-filtered frontier):")
for round_num in range(3):
    if not sorted_frontier_filtered:
        print(f"  Round {round_num}: WALK (no frontier offers beat outside option)")
        continue
    idx = round_num % len(sorted_frontier_filtered)
    my, opp = sorted_frontier_filtered[idx]
    fracs = [f"{opp[j]}/{QUANTITIES[j]}" for j in range(3)]
    print(f"  Round {round_num}: give {opp} (fractions: {fracs}) -- total {sum(opp)} items, my_val={EF1Agent._value_of(values, my):.0f}")

Cycling through rounds (using BATNA-filtered frontier):
  Round 0: give (3, 4, 1) (fractions: ['3/7', '4/4', '1/1']) -- total 8 items, my_val=320
  Round 1: give (4, 3, 0) (fractions: ['4/7', '3/4', '0/1']) -- total 7 items, my_val=310
  Round 2: give (4, 0, 1) (fractions: ['4/7', '0/4', '1/1']) -- total 5 items, my_val=320


## Step 6: Accepting and Walking

When the agent receives an offer:
- **Accept** if the offer is EF1 for the agent AND the value **strictly exceeds** the outside option
- **Walk** if the offer is EF1 but at or below the outside option
- **Counter** if the offer is not EF1 and counteroffers are still possible
- **Walk** if the offer is not EF1 and no counteroffer is possible (final round)
- **Walk** if the outside option dominates all EF1 allocations

In [45]:
# Simulate accept/reject decisions
outside_option = 300.0  # raw value


test_offers = [
    ((4, 2, 1), (3, 2, 0)),  # EF1, value 410 > 300
    ((3, 4, 1), (4, 0, 0)),  # EF1, value 370 > 300
    ((2, 4, 1), (5, 0, 0)),  # Not EF1
    ((2, 1, 0), (5, 3, 1)),  # Not EF1 (too generous to opp)
    ((3, 2, 0), (4, 2, 1)),  # Not EF1, value 280 < 300
]

print(f"Outside option: {outside_option}")
print(f"Acceptance: > {outside_option} ")
print()
for my, opp in test_offers:
    my_val = EF1Agent._value_of(values, my)
    ef1 = EF1Agent._is_ef1(values, my, opp)
    if ef1 and my_val > outside_option:
        decision = "ACCEPT"
    elif ef1:
        decision = "WALK (EF1 but not strictly above outside option)"
    else:
        decision = "COUNTER (not EF1)"
    print(f"  Receive {my}, opp keeps {opp}: val={my_val:.0f} EF1={ef1} -> {decision}")

Outside option: 300.0
Acceptance: > 300.0 

  Receive (4, 2, 1), opp keeps (3, 2, 0): val=410 EF1=True -> ACCEPT
  Receive (3, 4, 1), opp keeps (4, 0, 0): val=370 EF1=True -> ACCEPT
  Receive (2, 4, 1), opp keeps (5, 0, 0): val=290 EF1=False -> COUNTER (not EF1)
  Receive (2, 1, 0), opp keeps (5, 3, 1): val=180 EF1=False -> COUNTER (not EF1)
  Receive (3, 2, 0), opp keeps (4, 2, 1): val=280 EF1=False -> COUNTER (not EF1)


## Comparing Value Profiles

The agent's behavior adapts to different value profiles. Let's see how the frontier changes.

In [46]:
profiles = [
    ([50, 50, 50], "Equal values"),
    ([99, 1, 1], "Item 0 dominant"),
    ([1, 99, 1], "Item 1 dominant"),
    ([10, 10, 99], "Item 2 dominant"),
    ([80, 20, 50], "Mixed values"),
]

for vals, label in profiles:
    ef1 = agent._find_ef1_allocations(vals)
    frontier = EF1Agent._pareto_frontier(ef1)
    sf = EF1Agent._sort_frontier(frontier)
    opening_my, opening_opp = sf[0]
    print(f"{label} {vals}:")
    print(f"  {len(ef1)} EF1 allocations, {len(sf)} on frontier")
    print(f"  Opening: give {opening_opp} (total {sum(opening_opp)} items), keep {opening_my} (val={EF1Agent._value_of(vals, opening_my):.0f})")
    if len(sf) > 1:
        print(f"  Cycle options: {[opp for _, opp in sf[:3]]}")
    print()

Equal values [50, 50, 50]:
  45 EF1 allocations, 10 on frontier
  Opening: give (6, 0, 0) (total 6 items), keep (1, 4, 1) (val=300)
  Cycle options: [(6, 0, 0), (5, 1, 0), (4, 2, 0)]

Item 0 dominant [99, 1, 1]:
  45 EF1 allocations, 3 on frontier
  Opening: give (3, 4, 1) (total 8 items), keep (4, 0, 0) (val=396)
  Cycle options: [(3, 4, 1), (4, 2, 0), (4, 1, 1)]

Item 1 dominant [1, 99, 1]:
  48 EF1 allocations, 1 on frontier
  Opening: give (7, 2, 1) (total 10 items), keep (0, 2, 0) (val=198)

Item 2 dominant [10, 10, 99]:
  59 EF1 allocations, 7 on frontier
  Opening: give (7, 3, 0) (total 10 items), keep (0, 1, 1) (val=109)
  Cycle options: [(7, 3, 0), (6, 4, 0), (1, 4, 1)]

Mixed values [80, 20, 50]:
  45 EF1 allocations, 3 on frontier
  Opening: give (3, 4, 1) (total 8 items), keep (4, 0, 0) (val=320)
  Cycle options: [(3, 4, 1), (4, 3, 0), (4, 0, 1)]



## Action Encoding

The CUDA environment has an asymmetric action encoding: action `[n0, n1, n2]` always sets `current_offer` to items allocated to Player 2.

- **P1 action** `[n0,n1,n2]` = P2 gets these items, P1 keeps the rest
- **P2 action** `[n0,n1,n2]` = P2 keeps these items, P1 gets the rest

The agent handles this by encoding differently based on which player it is.

In [47]:
my_items = (4, 2, 0)
opp_items = (3, 2, 1)

# As P1: encode opponent's items (they are P2)
action_p1 = EF1Agent._allocation_to_action(my_items, opp_items, current_player=0)
print(f"Agent is P1, keeps {my_items}, gives {opp_items} to P2")
print(f"  Action = {opp_items[0]}*10 + {opp_items[1]}*2 + {opp_items[2]} = {action_p1}")
print()

# As P2: encode own items (agent IS P2)
action_p2 = EF1Agent._allocation_to_action(my_items, opp_items, current_player=1)
print(f"Agent is P2, keeps {my_items}, gives {opp_items} to P1")
print(f"  Action = {my_items[0]}*10 + {my_items[1]}*2 + {my_items[2]} = {action_p2}")

Agent is P1, keeps (4, 2, 0), gives (3, 2, 1) to P2
  Action = 3*10 + 2*2 + 1 = 35

Agent is P2, keeps (4, 2, 0), gives (3, 2, 1) to P1
  Action = 4*10 + 2*2 + 0 = 44
